In [3]:
import os
from datetime import datetime
from pathlib import Path
import torch


# 🚀 Включаем оптимизации CUDA
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True

# Если фиксированные размеры — ускорит сильно
torch.backends.cudnn.deterministic = False
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import json
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

In [4]:

# ============================================
# 📁 ФУНКЦИЯ СОЗДАНИЯ ПАПКИ ДЛЯ РЕЗУЛЬТАТОВ
# ============================================



def create_results_folder(epochs, batch_size, learning_rate, model_type='CNN_LSTM', base_dir='results'):
    """Создаёт папку для сохранения результатов эксперимента"""
    folder_name = f"{model_type}_{epochs}epochs_{batch_size}batch_{learning_rate}lr"
    results_dir = Path(base_dir) / folder_name
    results_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n📁 Папка результатов: {results_dir.absolute()}")
    return results_dir, folder_name

In [5]:

# ============================================
# 📊 PIPE DATASET (с нормализацией)
# ============================================

class PipeDataset(Dataset):
    def __init__(self, data_list, labels, scaler=None, fit_scaler=False):
        if fit_scaler:
            all_data = np.vstack(data_list)
            from sklearn.preprocessing import StandardScaler
            self.scaler = StandardScaler().fit(all_data)
        else:
            self.scaler = scaler
        
        self.data_list = [torch.FloatTensor(self.scaler.transform(sample)) 
                         for sample in data_list]
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.data_list)
    
    def __getitem__(self, idx):
        return self.data_list[idx], self.labels[idx]

def collate_fn(batch):
    """Формирует батч с padding для переменной длины"""
    data_list, labels = zip(*batch)
    max_len = max([d.shape[0] for d in data_list])
    
    padded_data = []
    lengths = []
    
    for data in data_list:
        length = data.shape[0]
        lengths.append(length)
        padding = torch.zeros(max_len - length, data.shape[1])
        padded = torch.cat([data, padding], dim=0)
        padded_data.append(padded)
    
    return torch.stack(padded_data), torch.stack(labels), torch.tensor(lengths)

In [6]:

# ============================================
# 📈 ФУНКЦИИ ОБУЧЕНИЯ И ОЦЕНКИ
# ============================================

def train_epoch(model, loader, criterion, optimizer, device, scaler):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for data, labels, lengths in loader:
        data = data.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        lengths = lengths.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():  # 🔥
            outputs = model(data, lengths)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * data.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return total_loss / total, 100. * correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for data, labels, lengths in loader:
            data = data.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            lengths = lengths.to(device, non_blocking=True)

            with torch.cuda.amp.autocast():
                outputs = model(data, lengths)
                loss = criterion(outputs, labels)

            total_loss += loss.item() * data.size(0)
            _, predicted = outputs.max(1)

            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / total, 100. * correct / total, all_preds, all_labels

In [7]:

# ============================================
# 📈 ФУНКЦИЯ ПОСТРОЕНИЯ ГРАФИКОВ
# ============================================

def plot_training_history(history, results_dir, folder_prefix, save_name='training_plot.png'):
    import matplotlib.pyplot as plt
    
    plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # График потерь
    axes[0].plot(epochs, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
    axes[0].plot(epochs, history['val_loss'], 'r--', label='Val Loss', linewidth=2)
    axes[0].set_xlabel('Эпоха', fontsize=11)
    axes[0].set_ylabel('Потеря (Loss)', fontsize=11)
    axes[0].set_title('📉 Динамика потери', fontsize=13, fontweight='bold')
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3)
    axes[0].set_axisbelow(True)
    
    # График точности
    axes[1].plot(epochs, history['train_acc'], 'b-', label='Train Acc', linewidth=2)
    axes[1].plot(epochs, history['val_acc'], 'r--', label='Val Acc', linewidth=2)
    axes[1].set_xlabel('Эпоха', fontsize=11)
    axes[1].set_ylabel('Точность (%)', fontsize=11)
    axes[1].set_title('📈 Динамика точности', fontsize=13, fontweight='bold')
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)
    axes[1].set_axisbelow(True)
    
    plt.tight_layout()
    save_path = results_dir / f"{folder_prefix}_{save_name}"
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"💾 График сохранён: {save_path}")

In [8]:

# ============================================
# === ХРАНИЛИЩЕ ИСТОРИИ ===
# ============================================

history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}




In [9]:

# ============================================
#  МОДЕЛЬ CNN + LSTM (HYBRID)
# ============================================
cnn_filters=[32, 64, 512, 128]

class PipeCNNLSTM(nn.Module):
    def __init__(self, input_channels=8, n_classes=2, 
                 cnn_filters=[32, 64, 128], 
                 lstm_hidden=128, 
                 lstm_layers=4,
                 dropout=0.3):
        super().__init__()
        
        # === CNN БЛОК (извлекает локальные признаки) ===
        cnn_layers = []
        in_channels = input_channels
        
        for filters in cnn_filters:
            cnn_layers.extend([
                nn.Conv1d(in_channels, filters, kernel_size=5, padding=2),
                nn.BatchNorm1d(filters),
                nn.ReLU(),
                nn.MaxPool1d(2),  
                nn.Dropout(dropout)
            ])
            in_channels = filters
        
        self.cnn = nn.Sequential(*cnn_layers)
        
        # LSTM БЛОК 
        self.lstm = nn.LSTM(
            input_size=cnn_filters[-1],
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,  # Двусторонний LSTM
            dropout=dropout if lstm_layers > 1 else 0
        )
        
        # === Классификатор ===
        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden * 2, 64),  # *2 для bidirectional
            #nn.ReLU(),
            nn.LeakyReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes)
        )
    
    def forward(self, x, lengths=None):
        # x: (B, L, C)
    
        # === CNN ===
        x = x.transpose(1, 2)
        x = self.cnn(x)
        x = x.transpose(1, 2)
    
        # === корректировка длин ===
        if lengths is not None:
            lengths = lengths.clone()
    
            num_pools = len(cnn_filters)

            for _ in range(num_pools):  # число MaxPool
                lengths = (lengths + 1) // 2
    
            lengths = torch.clamp(lengths, max=x.shape[1])
    
        # === PACK ===
        if lengths is not None:
            lengths_cpu = lengths.cpu()
    
            packed = pack_padded_sequence(
                x,
                lengths_cpu,
                batch_first=True,
                enforce_sorted=False
            )
    
            packed_out, (h_n, _) = self.lstm(packed)
    
            lstm_out, _ = pad_packed_sequence(
                packed_out,
                batch_first=True
            )
        else:
            lstm_out, (h_n, _) = self.lstm(x)
    
        # === ВЫХОД ===
        if self.lstm.bidirectional:
            h_cat = torch.cat([h_n[-2], h_n[-1]], dim=1)
        else:
            h_cat = h_n[-1]
    
        return self.classifier(h_cat)


In [10]:

# ============================================
# === КОНФИГУРАЦИЯ ===
# ============================================

#DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print("🔥 GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

BATCH_SIZE = 32  # Уменьшили для LSTM (требует больше памяти)
EPOCHS = 200     # Увеличили (LSTM обучается медленнее)
LEARNING_RATE = 0.001  # Оптимально для Adam + LSTM
N_CLASSES = 2
MODEL_TYPE = 'CNN_LSTM'

# 📁 Создаём папку для результатов
results_dir, folder_prefix = create_results_folder(
    epochs=EPOCHS, 
    batch_size=BATCH_SIZE, 
    learning_rate=LEARNING_RATE,
    model_type=MODEL_TYPE,
    base_dir='results'
)

🔥 GPU: CPU

📁 Папка результатов: /mnt/d/Универ/ВКР/Код/ПарсингДаных/DataTransform/scientificProject/results/CNN_LSTM_200epochs_32batch_0.001lr


/home/strel/miniconda3/lib/python3.13/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12060). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [11]:

# ============================================
# 📊 ПОДГОТОВКА DATASET И DATALOADER
# ============================================

train_dataset = PipeDataset(X_train, y_train, fit_scaler=True)
val_dataset = PipeDataset(X_val, y_val, scaler=train_dataset.scaler)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,              # ⬅️ увеличили
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=4,               # ⬅️ ВАЖНО
    pin_memory=True,             # ⬅️ ВАЖНО
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

print(f"\n✅ DataLoader создан!")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches:   {len(val_loader)}")

NameError: name 'X_train' is not defined

In [ ]:

# ============================================
# === МОДЕЛЬ ===
# ============================================

model = PipeCNNLSTM(
    input_channels=8, 
    n_classes=N_CLASSES,
    cnn_filters=[32, 64, 512, 128],
    lstm_hidden=128,
    lstm_layers=4,
    dropout=0.2  # Увеличенный dropout для борьбы с переобучением
).to(DEVICE)

criterion = nn.CrossEntropyLoss()

# Class weights для несбалансированных данных
# from sklearn.utils.class_weight import compute_class_weight
# class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
# criterion = nn.CrossEntropyLoss(weight=torch.FloatTensor(class_weights).to(DEVICE))

optimizer = optim.Adamax(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='max', 
    patience=10,    # Увеличили patience для LSTM
    factor=0.5,
    min_lr=1e-6
)

print(f"\n🚀 Старт обучения на {DEVICE}")
print(f"📊 Размеров параметров: {sum(p.numel() for p in model.parameters()):,}")
print(f"📁 Результаты будут сохранены в: {results_dir}")


In [ ]:


# ============================================
# === ЦИКЛ ОБУЧЕНИЯ ===
# ============================================

best_val_acc = 0
patience_counter = 0
early_stop_patience = 20  # Early stopping
scaler = torch.cuda.amp.GradScaler()

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE, scaler)
    val_loss, val_acc, val_preds, val_labels = evaluate(model, val_loader, criterion, DEVICE)
    
    # Сохраняем метрики
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    scheduler.step(val_acc)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), results_dir / 'best_pipe_cnn_lstm.pth')
        print(f"  ⭐ Новая лучшая модель! Acc: {val_acc:.2f}%")
    else:
        patience_counter += 1
    
    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}%")
    
    # Early Stopping
    if patience_counter >= early_stop_patience:
        print(f"\n🛑 Early stopping на эпохе {epoch+1} (нет улучшений {early_stop_patience} эпох)")
        break

In [ ]:

# ============================================
# === ФИНАЛЬНЫЙ ОТЧЕТ ===
# ============================================

print("\n" + "="*60)
print(f"✅ Лучшая точность валидации: {best_val_acc:.2f}%")
print(f"🏁 Обучение завершено после {len(history['train_acc'])} эпох")
print("="*60)

# 📊 Строим графики
print("\n📈 Строим графики обучения...")
plot_training_history(history, results_dir, folder_prefix, save_name='training_plot.png')

# 📋 Отчёт по классификации
try:
    from sklearn.metrics import classification_report, confusion_matrix
    import seaborn as sns
    import matplotlib.pyplot as plt
    
    print("\n📋 Classification Report:")
    print(classification_report(val_labels, val_preds, 
                             target_names=['Годный (0)', 'Брак (1)']))
    
    cm = confusion_matrix(val_labels, val_preds)
    print("\n🗂️ Confusion Matrix:")
    print(cm)
    
    # Визуализация матрицы ошибок
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Годный', 'Брак'],
                yticklabels=['Годный', 'Брак'])
    plt.xlabel('Предсказано', fontsize=11)
    plt.ylabel('Фактически', fontsize=11)
    plt.title('🎯 Матрица ошибок', fontsize=13, fontweight='bold')
    plt.tight_layout()
    
    cm_save_path = results_dir / f"{folder_prefix}_confusion_matrix.png"
    plt.savefig(cm_save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"💾 Матрица ошибок сохранена: {cm_save_path}")
    
    # Текстовый отчёт
    report_path = results_dir / f"{folder_prefix}_classification_report.txt"
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write("="*60 + "\n")
        f.write(f"Отчёт о классификации CNN+LSTM\n")
        f.write(f"Дата: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("="*60 + "\n\n")
        f.write(f"Параметры модели:\n")
        f.write(f"  - Модель: {MODEL_TYPE}\n")
        f.write(f"  - Эпохи: {EPOCHS}\n")
        f.write(f"  - Batch Size: {BATCH_SIZE}\n")
        f.write(f"  - Learning Rate: {LEARNING_RATE}\n")
        f.write(f"  - LSTM Hidden: 128\n")
        f.write(f"  - LSTM Layers: 2\n")
        f.write(f"  - Лучшая точность: {best_val_acc:.2f}%\n\n")
        f.write("Classification Report:\n")
        f.write(classification_report(val_labels, val_preds, 
                                     target_names=['Годный (0)', 'Брак (1)']))
        f.write("\nConfusion Matrix:\n")
        f.write(str(cm))
    
    print(f"💾 Текстовый отчёт сохранён: {report_path}")
    
except ImportError:
    print("\n⚠️ sklearn не установлен")
    print(f"Простая точность: {np.mean(np.array(val_preds) == np.array(val_labels)) * 100:.2f}%")

In [ ]:

# ============================================
# 📄 СОХРАНЕНИЕ КОНФИГУРАЦИИ
# ============================================

config_path = results_dir / f"{folder_prefix}_config.json"

config = {
    'model_type': MODEL_TYPE,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'n_classes': N_CLASSES,
    'input_channels': 8,
    'cnn_filters': [32, 64, 128],
    'lstm_hidden': 128,
    'lstm_layers': 2,
    'dropout': 0.4,
    'best_val_acc': best_val_acc,
    'train_samples': len(X_train),
    'val_samples': len(X_val),
    'actual_epochs': len(history['train_acc']),
    'timestamp': datetime.now().isoformat(),
    'device': str(DEVICE)
}

with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print(f"💾 Конфигурация сохранена: {config_path}")

# ============================================
# 📋 ИТОГОВАЯ ИНФОРМАЦИЯ
# ============================================

print("\n" + "="*60)
print("📁 ВСЕ ФАЙЛЫ СОХРАНЕНЫ В ПАПКУ:")
print(f"   {results_dir.absolute()}")
print("\n📄 Список сохранённых файлов:")
print(f"   1. {folder_prefix}_best_pipe_cnn_lstm.pth (веса модели)")
print(f"   2. {folder_prefix}_training_plot.png (графики)")
print(f"   3. {folder_prefix}_confusion_matrix.png (матрица ошибок)")
print(f"   4. {folder_prefix}_classification_report.txt (отчёт)")
print(f"   5. {folder_prefix}_config.json (параметры)")
print("="*60)import os
from datetime import datetime
from pathlib import Path
import torch

